In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Model and Loss

In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def masked_timewise_mean_std(x: torch.Tensor, m_hat: torch.Tensor, eps: float = 1e-5):
    """
    Compute mean/std for each sample and variable across time only.

    x     : (B, T, D)
    m_hat : (B, T, D), 1=observed, 0=missing

    Returns:
      mu    : (B, 1, D)
      sigma : (B, 1, D)
    """
    m = m_hat.to(dtype=x.dtype)

    count = m.sum(dim=1, keepdim=True).clamp_min(1.0)          # (B,1,D)
    mu = (x * m).sum(dim=1, keepdim=True) / count              # (B,1,D)

    var = (((x - mu) * m) ** 2).sum(dim=1, keepdim=True) / count
    sigma = torch.sqrt(var + eps)
    return mu, sigma


class LearnablePositionalEncoding(nn.Module):
    """
    Fully learnable positional encoding E_pos in R^(T x L)
    """
    def __init__(self, seq_len: int, d_model: int, init_std: float = 0.02):
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.normal_(self.pe, mean=0.0, std=init_std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        return x + self.pe[:, :T, :]


class TimeBatchNorm(nn.Module):
    """
    BatchNorm over latent dimension L for sequences shaped (B,T,L).
    Implemented via BatchNorm1d(L) on permuted tensor (B,L,T).
    """
    def __init__(self, d_model: int):
        super().__init__()
        self.bn = nn.BatchNorm1d(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,T,L)
        x = x.transpose(1, 2)      # (B,L,T)
        x = self.bn(x)
        x = x.transpose(1, 2)      # (B,T,L)
        return x

class MTSITEncoderBlock(nn.Module):
    """
    Paper-style encoder block:
      X_bn  = BN(X + MHA(X))
      X_ffn = Dropout( GELU( X_bn W_h ) W_o )
      Y     = BN(X_bn + X_ffn)
    """
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_ff: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.mha = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.bn1 = TimeBatchNorm(d_model)
        self.bn2 = TimeBatchNorm(d_model)

        self.ffn_fc1 = nn.Linear(d_model, d_ff)
        self.ffn_fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):
        # x: (B,T,L)

        # MHA + residual + BN
        attn_out, attn_weights = self.mha(x, x, x, need_weights=True)
        x_bn = self.bn1(x + attn_out)

        # FFN(GELU) + residual + BN
        x_ffn = self.ffn_fc2(F.gelu(self.ffn_fc1(x_bn)))
        x_ffn = self.dropout(x_ffn)

        y = self.bn2(x_bn + x_ffn)
        return y, attn_weights


class MTSITImputerOriginalAligned(nn.Module):
    """
    Inputs:
      x_in  : (B,T,D)   missing values already prefilled (e.g. 0)
      m_hat : (B,T,D)   1=observed, 0=missing

    Outputs:
      x_hat      : (B,T,D)  reconstructed / predicted sequence
      x_c        : (B,T,D)  complemented imputed series
      x_norm_hat : (B,T,D)  predicted normalized output before de-normalization
      mu, sigma  : (B,1,D)
    """
    def __init__(
        self,
        seq_len: int = 1080,
        d_in: int = 2,
        d_model: int = 128,
        n_heads: int = 8,
        d_ff: int = 256,
        n_layers: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.seq_len = seq_len
        self.d_in = d_in
        self.d_model = d_model

        # Eq. (1): X_p = X_norm W_p + B_p + E_pos
        self.in_proj = nn.Linear(d_in, d_model, bias=False)   # W_p
        self.input_bias = nn.Parameter(torch.zeros(1, seq_len, d_model))  # B_p
        nn.init.normal_(self.input_bias, mean=0.0, std=0.02)

        self.pos = LearnablePositionalEncoding(seq_len=seq_len, d_model=d_model)

        self.encoder_blocks = nn.ModuleList([
            MTSITEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        # Eq. (10): X_hat = Z W_o + B_o
        self.out_proj = nn.Linear(d_model, d_in, bias=True)

    def normalize_inputs(self, x_in: torch.Tensor, m_hat: torch.Tensor, eps: float = 1e-5):
        """
        Per-sample, per-variable normalization across sequence length.
        Uses only observed values to estimate mean/std.
        """
        mu, sigma = masked_timewise_mean_std(x_in, m_hat, eps=eps)
        x_norm = (x_in - mu) / sigma
        x_norm = m_hat * x_norm
        return x_norm, mu, sigma

    def forward(self, x_in: torch.Tensor, m_hat: torch.Tensor, return_attn: bool = False):
        B, T, D = x_in.shape
        assert T == self.seq_len and D == self.d_in, \
            f"Expected (B,{self.seq_len},{self.d_in}), got {tuple(x_in.shape)}"
        assert m_hat.shape == x_in.shape, "m_hat must have same shape as x_in"
        x_norm, mu, sigma = self.normalize_inputs(x_in, m_hat)
        h = self.in_proj(x_norm) + self.input_bias[:, :T, :]
        h = self.pos(h)

        attn_list = []
        for block in self.encoder_blocks:
            h, attn_w = block(h)
            if return_attn:
                attn_list.append(attn_w)

        x_norm_hat = self.out_proj(h)                  # (B,T,D)
        x_hat = x_norm_hat * sigma + mu                # (B,T,D)
        x_c = m_hat * x_in + (1.0 - m_hat) * x_hat

        if return_attn:
            return x_hat, x_c, x_norm_hat, mu, sigma, attn_list
        return x_hat, x_c, x_norm_hat, mu, sigma

### Model training, validation, and testing

In [4]:
NPZ_PATH = "/content/drive/MyDrive/aim3/saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz"
OUT_DIR  = "/content/drive/MyDrive/aim3/mtsit_original_aligned_1e-3_64"

In [5]:

import os
import math
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# Config
# -----------------------------

CKPT_DIR = os.path.join(OUT_DIR, "ckpts")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

BATCH_SIZE   = 64
EPOCHS       = 300
LR           = 1e-3
WEIGHT_DECAY = 1e-4
LAMBDA_MIT   = 1.0
LAMBDA_ORT   = 0.0
NUM_WORKERS  = 0
SEED         = 1337

N_SPLITS = 5
VAL_WITHIN_TRAIN_P = 0.10
EARLY_STOPPING_PATIENCE = 15

# Transformer hyperparams
D_IN = 2      
SEQ_LEN = 1080 
D_MODEL = 16
N_HEADS = 4
D_FF = 16
N_LAYERS = 3
DROPOUT  = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = (DEVICE == "cuda")
print("DEVICE:", DEVICE)


# ============================================================
# Seed
# ============================================================
def seed_all(seed=1337):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)


# ============================================================
# Helpers
# ============================================================
def rmse_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))


def compute_gap_lengths_from_mask(mask_1d, threshold=0.5):
    """
    For each time point in a masked run, assign the full contiguous run length.
    Non-masked points get 0.
    """
    mask_bin = (np.asarray(mask_1d) > threshold).astype(np.int32)
    T = len(mask_bin)
    gap_len = np.zeros(T, dtype=np.int32)

    start = 0
    while start < T:
        if mask_bin[start] == 0:
            start += 1
            continue

        end = start
        while end + 1 < T and mask_bin[end + 1] == 1:
            end += 1

        run_len = end - start + 1
        gap_len[start:end + 1] = run_len
        start = end + 1

    return gap_len


def assign_gap_region(g):
    g = int(g)
    if 1 <= g < 12:
        return "1-12"
    elif 13 <= g < 120:
        return "12-120"
    elif 121 <= g < 240:
        return "120-240"
    elif 241 <= g <= 360:
        return "240-360"
    else:
        return None


def make_patient_folds(unique_patients, n_splits=5, seed=1337):
    pats = np.array(unique_patients)
    rng = np.random.default_rng(seed)
    rng.shuffle(pats)
    folds = np.array_split(pats, n_splits)
    return [np.array(f, dtype=pats.dtype) for f in folds]


def split_train_val_patients(trainval_patients, val_fraction=0.10, seed=1337):
    trainval_patients = np.array(trainval_patients)
    n_trainval = len(trainval_patients)

    n_val = max(1, int(round(n_trainval * val_fraction)))
    if n_val >= n_trainval:
        n_val = n_trainval - 1

    rng = np.random.default_rng(seed)
    shuffled = trainval_patients.copy()
    rng.shuffle(shuffled)

    val_pats = shuffled[:n_val]
    train_pats = shuffled[n_val:]
    return np.array(train_pats), np.array(val_pats)


def patient_ids_to_segment_indices(patient_ids_all, selected_patients):
    selected_patients = set(map(int, selected_patients))
    idx = np.array([i for i, p in enumerate(patient_ids_all) if int(p) in selected_patients], dtype=np.int64)
    return idx


# ============================================================
# Losses
# ============================================================
def masked_mse(estimation, target, mask, eps=1e-8):
    mask = mask.to(dtype=estimation.dtype)
    num = (((estimation - target) ** 2) * mask).sum()
    den = mask.sum().clamp_min(eps)
    return num / den


def mtsit_losses(x_hat, x_c, x_true, m_hat, i_mask,
                 lam_ort=0.0, lam_mit=1.0):
    mask_blank = ((m_hat == 1.0) & (i_mask == 0.0)).float()
    L_ort = masked_mse(x_hat, x_true, mask_blank)
    mask_art = (i_mask == 1.0).float()
    L_mit = masked_mse(x_hat, x_true, mask_art)
    L = lam_ort * L_ort + lam_mit * L_mit
    return L, L_ort, L_mit


# ============================================================
# Dataset
# ============================================================
class TSImputeDataset(Dataset):
    def __init__(self, X_in, M_hat, X_true, I_mask, seg_indices):
        X_true = np.nan_to_num(X_true, nan=0.0).astype(np.float32)

        self.X_in      = torch.from_numpy(X_in.astype(np.float32))
        self.M_hat     = torch.from_numpy(M_hat.astype(np.float32))
        self.X_true    = torch.from_numpy(X_true.astype(np.float32))
        self.I_mask    = torch.from_numpy(I_mask.astype(np.float32))
        self.seg_index = torch.from_numpy(seg_indices.astype(np.int64))

        assert self.X_in.shape == self.M_hat.shape == self.X_true.shape == self.I_mask.shape
        assert self.X_in.ndim == 3 and self.X_in.shape[-1] == D_IN

    def __len__(self):
        return self.X_in.shape[0]

    def __getitem__(self, idx):
        return (
            self.X_in[idx],
            self.M_hat[idx],
            self.X_true[idx],
            self.I_mask[idx],
            self.seg_index[idx],
        )


# ============================================================
# Load NPZ
# ============================================================
data = np.load(NPZ_PATH, allow_pickle=True)

X_in   = data["X_in"]        # (N,T,D)
M_hat  = data["M_hat"]       # (N,T,D)
X_true = data["X"]           # (N,T,D)
I_mask = data["I"]           # (N,T,D)
seg_id = data["segment_ids"] # (N,)

if "filenames" in data.files:
    filenames = np.array([str(x) for x in data["filenames"].tolist()])
else:
    filenames = np.array([f"segment_{i}" for i in range(len(seg_id))])

N, T, D = X_in.shape

if "target_columns" in data.files and len(data["target_columns"]) == D:
    target_names = [str(x) for x in data["target_columns"].tolist()]
elif "columns_all" in data.files and len(data["columns_all"]) == D:
    target_names = [str(x) for x in data["columns_all"].tolist()]
else:
    target_names = [f"channel_{d}" for d in range(D)]

print("Shapes:")
print("  X_in   :", X_in.shape)
print("  M_hat  :", M_hat.shape)
print("  X_true :", X_true.shape)
print("  I_mask :", I_mask.shape)
print("  seg_id :", seg_id.shape)
print("Target names:", target_names)
print("Unique patients:", len(np.unique(seg_id)))

assert D == D_IN, f"Expected D={D_IN}, got D={D}"
assert T == SEQ_LEN, f"Expected T={SEQ_LEN}, got T={T}"


# ============================================================
# Epoch runner
# ============================================================
def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)

    tot_L = 0.0
    tot_ort = 0.0
    tot_mit = 0.0
    n_batches = 0

    for x_in_b, m_hat_b, x_true_b, i_mask_b, _seg_idx_b in loader:
        x_in_b   = x_in_b.to(DEVICE, non_blocking=True)
        m_hat_b  = m_hat_b.to(DEVICE, non_blocking=True)
        x_true_b = x_true_b.to(DEVICE, non_blocking=True)
        i_mask_b = i_mask_b.to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        x_hat, x_c, x_norm_hat, mu, sigma = model(x_in_b, m_hat_b)

        L, L_ort, L_mit = mtsit_losses(
            x_hat=x_hat,
            x_c=x_c,
            x_true=x_true_b,
            m_hat=m_hat_b,
            i_mask=i_mask_b,
            lam_ort=LAMBDA_ORT,
            lam_mit=LAMBDA_MIT
        )

        if train:
            L.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        tot_L   += float(L.detach().cpu())
        tot_ort += float(L_ort.detach().cpu())
        tot_mit += float(L_mit.detach().cpu())
        n_batches += 1

    return tot_L / max(n_batches, 1), tot_ort / max(n_batches, 1), tot_mit / max(n_batches, 1)


# ============================================================
# Collect pointwise test predictions
# ============================================================
@torch.no_grad()
def collect_test_point_predictions(model, loader, patient_ids_all, filenames_all, target_names):
    """
    Collect pointwise predictions ONLY on artificially masked points (I_mask == 1).
    Use x_c for MIT evaluation because x_c is the completed output:
        x_c = M_hat * x_in + (1 - M_hat) * x_hat
    """
    model.eval()
    rows = []

    for x_in_b, m_hat_b, x_true_b, i_mask_b, seg_idx_b in loader:
        x_in_b   = x_in_b.to(DEVICE, non_blocking=True)
        m_hat_b  = m_hat_b.to(DEVICE, non_blocking=True)
        x_true_b = x_true_b.to(DEVICE, non_blocking=True)
        i_mask_b = i_mask_b.to(DEVICE, non_blocking=True)

        x_hat, x_c, x_norm_hat, mu, sigma = model(x_in_b, m_hat_b)

        pred_np = x_c.detach().cpu().numpy()
        true_np = x_true_b.detach().cpu().numpy()
        mask_np = i_mask_b.detach().cpu().numpy()
        seg_np  = seg_idx_b.detach().cpu().numpy()

        B = pred_np.shape[0]

        for b in range(B):
            seg_global = int(seg_np[b])

            for d in range(pred_np.shape[2]):
                gap_len_arr = compute_gap_lengths_from_mask(mask_np[b, :, d])

                masked_t = np.where(mask_np[b, :, d] > 0.5)[0]
                for t in masked_t:
                    g = int(gap_len_arr[t])
                    rows.append({
                        "segment_idx": seg_global,
                        "patient_id": int(patient_ids_all[seg_global]),
                        "filename": str(filenames_all[seg_global]),
                        "time_idx": int(t),
                        "target": str(target_names[d]),
                        "y_true": float(true_np[b, t, d]),
                        "y_pred": float(pred_np[b, t, d]),
                        "abs_err": float(abs(true_np[b, t, d] - pred_np[b, t, d])),
                        "gap_len": g,
                        "gap_region": assign_gap_region(g),
                    })

    return pd.DataFrame(rows)


# ============================================================
# Build patient folds
# ============================================================
patients = np.unique(seg_id)
patient_folds = make_patient_folds(patients, n_splits=N_SPLITS, seed=SEED)

print("\nPatient-level folds:")
for i, pats in enumerate(patient_folds, start=1):
    print(f"  Fold {i}: n_test_patients={len(pats)} | patients={sorted(map(int, pats.tolist()))}")


# ============================================================
# CV containers
# ============================================================
all_fold_rows = []
all_pred_rows = []
fold_patient_rows = []


# ============================================================
# 5-fold CV
# ============================================================
for fold, test_pats in enumerate(patient_folds, start=1):
    print("\n" + "#" * 120)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("#" * 120)

    test_pats = np.array(test_pats)
    trainval_pats = np.array([p for p in patients if p not in set(test_pats)])

    train_pats, val_pats = split_train_val_patients(
        trainval_patients=trainval_pats,
        val_fraction=VAL_WITHIN_TRAIN_P,
        seed=SEED + fold
    )

    train_idx = patient_ids_to_segment_indices(seg_id, train_pats)
    val_idx   = patient_ids_to_segment_indices(seg_id, val_pats)
    test_idx  = patient_ids_to_segment_indices(seg_id, test_pats)

    print(f"Patients: train={len(train_pats)}, val={len(val_pats)}, test={len(test_pats)}")
    print(f"Segments: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

    fold_patient_rows.append({
        "fold": fold,
        "train_patients": ",".join(map(str, sorted(map(int, train_pats.tolist())))),
        "val_patients": ",".join(map(str, sorted(map(int, val_pats.tolist())))),
        "test_patients": ",".join(map(str, sorted(map(int, test_pats.tolist())))),
        "n_train_patients": len(train_pats),
        "n_val_patients": len(val_pats),
        "n_test_patients": len(test_pats),
        "n_train_segments": len(train_idx),
        "n_val_segments": len(val_idx),
        "n_test_segments": len(test_idx),
    })

    # -----------------------------
    # DataLoaders
    # -----------------------------
    train_ds = TSImputeDataset(X_in[train_idx], M_hat[train_idx], X_true[train_idx], I_mask[train_idx], train_idx)
    val_ds   = TSImputeDataset(X_in[val_idx],   M_hat[val_idx],   X_true[val_idx],   I_mask[val_idx],   val_idx)
    test_ds  = TSImputeDataset(X_in[test_idx],  M_hat[test_idx],  X_true[test_idx],  I_mask[test_idx],  test_idx)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )

    # -----------------------------
    # Model
    # -----------------------------
    model = MTSITImputerOriginalAligned(
        seq_len=T,
        d_in=D_IN,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        d_ff=D_FF,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    ckpt_path = os.path.join(CKPT_DIR, f"fold{fold:02d}_best.pt")

    # -----------------------------
    # Train
    # -----------------------------
    best_val = float("inf")
    best_epoch = -1
    no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        trL, trO, trM = run_epoch(model, train_loader, optimizer=optimizer)
        vaL, vaO, vaM = run_epoch(model, val_loader, optimizer=None)
        scheduler.step(vaL)

        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch:03d} | lr={lr_now:.2e} | "
            f"TRAIN L={trL:.5f} (ORT={trO:.5f}, MIT={trM:.5f}) | "
            f"VAL L={vaL:.5f} (ORT={vaO:.5f}, MIT={vaM:.5f})"
        )

        if vaL < best_val:
            best_val = vaL
            best_epoch = epoch
            no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
            print("saved best weights:", ckpt_path)
        else:
            no_improve += 1
            if no_improve >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    print("Best epoch:", best_epoch, "best val:", best_val)

    # -----------------------------
    # Reload best
    # -----------------------------

    state = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()



    teL, teO, teM = run_epoch(model, test_loader, optimizer=None)
    print(f"TEST  L={teL:.5f} (ORT={teO:.5f}, MIT={teM:.5f})")

    # -----------------------------
    # Collect pointwise predictions
    # -----------------------------
    pred_df_fold = collect_test_point_predictions(
        model=model,
        loader=test_loader,
        patient_ids_all=seg_id,
        filenames_all=filenames,
        target_names=target_names,
    )

    pred_df_fold["fold"] = fold
    pred_df_fold["model"] = "MTSIT_OriginalAligned"

    all_pred_rows.append(pred_df_fold)

    # -----------------------------
    # Overall fold metrics by channel
    # -----------------------------
    for target, sdf in pred_df_fold.groupby("target"):
        fold_mae = mae_np(sdf["y_true"], sdf["y_pred"])
        fold_rmse = rmse_np(sdf["y_true"], sdf["y_pred"])

        all_fold_rows.append({
            "fold": fold,
            "target": target,
            "model": "MTSIT_OriginalAligned",
            "val_total_loss": best_val,
            "test_total_loss": teL,
            "test_mae": fold_mae,
            "test_rmse": fold_rmse,
            "n_points": len(sdf),
            "n_train_patients": len(train_pats),
            "n_val_patients": len(val_pats),
            "n_test_patients": len(test_pats),
            "n_train_segments": len(train_idx),
            "n_val_segments": len(val_idx),
            "n_test_segments": len(test_idx),
            "best_epoch": best_epoch,
        })


# ============================================================
# Combine results
# ============================================================
fold_df = pd.DataFrame(all_fold_rows)
pred_df = pd.concat(all_pred_rows, axis=0, ignore_index=True)
fold_patient_df = pd.DataFrame(fold_patient_rows)

fold_csv = os.path.join(OUT_DIR, "cv5_fold_results.csv")
pred_csv = os.path.join(OUT_DIR, "cv5_test_predictions_pooled.csv")
split_csv = os.path.join(OUT_DIR, "cv5_patient_split_summary.csv")

fold_df.to_csv(fold_csv, index=False)
pred_df.to_csv(pred_csv, index=False)
fold_patient_df.to_csv(split_csv, index=False)


# ============================================================
# Summary: mean ± std across folds
# ============================================================
summary_df = (
    fold_df
    .groupby(["target", "model"], as_index=False)
    .agg(
        val_total_loss_mean=("val_total_loss", "mean"),
        val_total_loss_std=("val_total_loss", "std"),
        test_total_loss_mean=("test_total_loss", "mean"),
        test_total_loss_std=("test_total_loss", "std"),
        test_mae_mean=("test_mae", "mean"),
        test_mae_std=("test_mae", "std"),
        test_rmse_mean=("test_rmse", "mean"),
        test_rmse_std=("test_rmse", "std"),
        n_folds=("fold", "count"),
        total_points=("n_points", "sum"),
    )
)

summary_csv = os.path.join(OUT_DIR, "cv5_summary_mean_std.csv")
summary_df.to_csv(summary_csv, index=False)


# ============================================================
# Pooled out-of-fold test metrics
# ============================================================
pooled_rows = []
for (target, model), sdf in pred_df.groupby(["target", "model"]):
    pooled_rows.append({
        "target": target,
        "model": model,
        "pooled_test_mae": mae_np(sdf["y_true"], sdf["y_pred"]),
        "pooled_test_rmse": rmse_np(sdf["y_true"], sdf["y_pred"]),
        "n_points": len(sdf),
    })

pooled_df = pd.DataFrame(pooled_rows)
pooled_csv = os.path.join(OUT_DIR, "cv5_pooled_test_metrics.csv")
pooled_df.to_csv(pooled_csv, index=False)


# ============================================================
# Gap-region summaries
# ============================================================
pred_df_gap = pred_df[pred_df["gap_region"].notna()].copy()

pred_gap_csv = os.path.join(OUT_DIR, "cv5_test_predictions_pooled_with_gap_region.csv")
pred_df_gap.to_csv(pred_gap_csv, index=False)

# pooled by gap region
pooled_gap_rows = []
for (target, model, gap_region), sdf in pred_df_gap.groupby(["target", "model", "gap_region"]):
    pooled_gap_rows.append({
        "target": target,
        "model": model,
        "gap_region": str(gap_region),
        "pooled_test_mae": mae_np(sdf["y_true"], sdf["y_pred"]),
        "pooled_test_rmse": rmse_np(sdf["y_true"], sdf["y_pred"]),
        "n_points": len(sdf),
    })

pooled_gap_df = pd.DataFrame(pooled_gap_rows)
pooled_gap_csv = os.path.join(OUT_DIR, "cv5_pooled_test_metrics_by_gap_region.csv")
pooled_gap_df.to_csv(pooled_gap_csv, index=False)

# fold-wise by gap region
fold_gap_rows = []
for (fold, target, model, gap_region), sdf in pred_df_gap.groupby(["fold", "target", "model", "gap_region"]):
    fold_gap_rows.append({
        "fold": fold,
        "target": target,
        "model": model,
        "gap_region": str(gap_region),
        "test_mae": mae_np(sdf["y_true"], sdf["y_pred"]),
        "test_rmse": rmse_np(sdf["y_true"], sdf["y_pred"]),
        "n_points": len(sdf),
    })

fold_gap_df = pd.DataFrame(fold_gap_rows)
fold_gap_csv = os.path.join(OUT_DIR, "cv5_fold_metrics_by_gap_region.csv")
fold_gap_df.to_csv(fold_gap_csv, index=False)

gap_summary_df = (
    fold_gap_df
    .groupby(["target", "model", "gap_region"], as_index=False)
    .agg(
        test_mae_mean=("test_mae", "mean"),
        test_mae_std=("test_mae", "std"),
        test_rmse_mean=("test_rmse", "mean"),
        test_rmse_std=("test_rmse", "std"),
        n_folds=("fold", "count"),
        total_points=("n_points", "sum"),
    )
)

gap_summary_csv = os.path.join(OUT_DIR, "cv5_summary_mean_std_by_gap_region.csv")
gap_summary_df.to_csv(gap_summary_csv, index=False)


# ============================================================
# Print final paths
# ============================================================
print("\nSaved:")
print("  fold results     :", fold_csv)
print("  pooled preds     :", pred_csv)
print("  pooled preds+gap :", pred_gap_csv)
print("  mean±std         :", summary_csv)
print("  pooled test      :", pooled_csv)
print("  patient splits   :", split_csv)
print("  pooled gap test  :", pooled_gap_csv)
print("  fold gap metrics :", fold_gap_csv)
print("  gap mean±std     :", gap_summary_csv)

print("\nPer-fold results:")
print(fold_df)

print("\nMean ± std summary:")
print(summary_df)

print("\nPooled out-of-fold test metrics:")
print(pooled_df)

print("\nPooled out-of-fold test metrics by gap region:")
print(pooled_gap_df)

print("\nMean ± std by gap region:")
print(gap_summary_df)

print("\nPatient split summary:")
print(fold_patient_df)

DEVICE: cuda
Shapes:
  X_in   : (3817, 1080, 2)
  M_hat  : (3817, 1080, 2)
  X_true : (3817, 1080, 2)
  I_mask : (3817, 1080, 2)
  seg_id : (3817,)
Target names: ['channel_0', 'channel_1']
Unique patients: 64

Patient-level folds:
  Fold 1: n_test_patients=13 | patients=[6, 13, 16, 22, 30, 31, 47, 52, 54, 59, 65, 68, 71]
  Fold 2: n_test_patients=13 | patients=[3, 9, 10, 11, 12, 15, 27, 33, 34, 36, 61, 67, 69]
  Fold 3: n_test_patients=13 | patients=[4, 19, 20, 24, 28, 38, 43, 44, 50, 56, 62, 64, 66]
  Fold 4: n_test_patients=13 | patients=[7, 8, 14, 17, 21, 25, 32, 45, 49, 51, 58, 63, 70]
  Fold 5: n_test_patients=12 | patients=[1, 2, 5, 23, 26, 35, 37, 48, 53, 55, 57, 60]

########################################################################################################################
FOLD 1/5
########################################################################################################################
Patients: train=46, val=5, test=13
Segments: train=2814, val=256,

### Model Validation

In [6]:

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
SPLIT_CSV = os.path.join(OUT_DIR, 'cv5_patient_split_summary.csv')
SEQ_LEN, D_IN = 1080, 2
D_MODEL, N_HEADS, D_FF, N_LAYERS, DROPOUT = 16, 4, 16, 3, 0.1
BATCH_SIZE = 256
SEED, N_SPLITS, VAL_WITHIN_TRAIN_P = 1337, 5, 0.10
LAMBDA_ORT, LAMBDA_MIT = 0.0, 1.0
EPS = 1e-8
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'MTSIT_OriginalAligned'
print('Device:', DEVICE)

if 'MTSITImputerOriginalAligned' not in globals():
    raise RuntimeError('Define MTSITImputerOriginalAligned exactly as during training before running this script.')
if not os.path.isdir(CKPT_DIR):
    raise FileNotFoundError(f'Checkpoint directory not found: {CKPT_DIR}')

class ValidationDataset(Dataset):
    def __init__(self, x_in, obs, truth, target, global_indices):
        self.x_in = torch.from_numpy(np.nan_to_num(x_in, nan=0, posinf=0, neginf=0).astype(np.float32))
        self.obs = torch.from_numpy(obs.astype(np.float32))
        self.truth = torch.from_numpy(np.nan_to_num(truth, nan=0, posinf=0, neginf=0).astype(np.float32))
        self.target = torch.from_numpy(target.astype(np.float32))
        self.ids = np.asarray(global_indices, dtype=np.int64)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return self.x_in[i], self.obs[i], self.truth[i], self.target[i], int(self.ids[i])

def parse_patients(v):
    if pd.isna(v) or str(v).strip() == '':
        return []
    return [int(float(s.strip())) for s in str(v).split(',') if s.strip()]

def reproduce_validation_splits(segment_ids):
    """Exact fallback from MTSIT training code when split CSV wasn't saved."""
    patients = np.unique(segment_ids)
    shuffled = patients.copy()
    np.random.default_rng(SEED).shuffle(shuffled)
    fold_patients = np.array_split(shuffled, N_SPLITS)
    records = []
    for fold, test_pats in enumerate(fold_patients, 1):
        trainval_pats = np.array([p for p in patients if p not in set(test_pats)])
        nval = max(1, min(len(trainval_pats)-1, round(len(trainval_pats) * VAL_WITHIN_TRAIN_P)))
        shuffled_tv = trainval_pats.copy()
        np.random.default_rng(SEED + fold).shuffle(shuffled_tv)
        val_pats = shuffled_tv[:nval]
        records.append(dict(fold=fold, val_patients=','.join(map(str, val_pats))))
    return pd.DataFrame(records)

def gap_lengths(mask):
    a = np.asarray(mask) > .5
    edge = np.flatnonzero(np.diff(np.r_[False, a, False].astype(np.int8)))
    lengths = np.zeros(a.size, dtype=np.int32)
    for s, e in zip(edge[::2], edge[1::2]):
        lengths[s:e] = e-s
    return lengths

def gap_region(g):
    # Match CSDI validation boundaries, including g=360.
    if 1 <= g <= 11: return '<1 min'
    if 12 <= g <= 119: return '[1,10) min'
    if 120 <= g <= 239: return '[10,20) min'
    if 240 <= g <= 359: return '[20,30) min'
    if g == 360: return '=30 min'
    return None

def metrics(df):
    if df.empty:
        return dict(mae=np.nan, rmse=np.nan, mape_pct=np.nan, n_points=0, n_mape=0)
    yt = df.y_true.to_numpy(dtype=np.float64)
    yp = df.y_pred.to_numpy(dtype=np.float64)
    err = yp-yt
    eligible = np.abs(yt) > EPS
    return dict(mae=float(np.abs(err).mean()),
                rmse=float(np.sqrt(np.square(err).mean())),
                mape_pct=float(100*np.mean(np.abs(err[eligible]/yt[eligible]))) if eligible.any() else np.nan,
                n_points=len(df), n_mape=int(eligible.sum()))

with np.load(NPZ_PATH, allow_pickle=True) as data:
    x_in = np.asarray(data['X_in'])
    observed = np.asarray(data['M_hat'])
    truth = np.asarray(data['X'])
    target = np.asarray(data['I'])
    seg_id = np.asarray(data['segment_ids'])
    filenames = np.asarray(data['filenames']).astype(str) if 'filenames' in data.files else np.array([f'segment_{i}' for i in range(len(seg_id))])
    if 'target_columns' in data.files and len(data['target_columns']) == D_IN:
        target_names = np.asarray(data['target_columns']).astype(str).tolist()
    elif 'columns_all' in data.files and len(data['columns_all']) == D_IN:
        target_names = np.asarray(data['columns_all']).astype(str).tolist()
    else:
        target_names = [f'channel_{d}' for d in range(D_IN)]

assert x_in.shape == observed.shape == truth.shape == target.shape
assert x_in.shape[1:] == (SEQ_LEN, D_IN)
assert len(seg_id) == len(x_in)
if np.any((observed > .5) & (target > .5)):
    raise ValueError('M_hat and I overlap')
if not np.isfinite(truth[target > .5]).all():
    raise ValueError('Ground truth must be finite at artificial validation targets')

if os.path.isfile(SPLIT_CSV):
    splits = pd.read_csv(SPLIT_CSV)
    print('Using saved patient splits:', SPLIT_CSV)
else:
    splits = reproduce_validation_splits(seg_id)
    print('Split CSV not found; exactly reconstructing the MTSIT validation split from seed 1337.')

all_rows, all_loss, all_fold_metrics, all_gap_metrics = [], [], [], []

for _, r in splits.iterrows():
    fold = int(r['fold'])
    vals = set(parse_patients(r['val_patients']))
    indices = np.flatnonzero(np.isin(seg_id, list(vals)))
    if not len(indices):
        raise ValueError(f'Fold {fold}: no validation segments')
    loader = DataLoader(ValidationDataset(x_in[indices], observed[indices], truth[indices], target[indices], indices),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(DEVICE=='cuda'))
    model = MTSITImputerOriginalAligned(
        seq_len=SEQ_LEN, d_in=D_IN, d_model=D_MODEL, n_heads=N_HEADS,
        d_ff=D_FF, n_layers=N_LAYERS, dropout=DROPOUT).to(DEVICE)
    ckpt = os.path.join(CKPT_DIR, f'fold{fold:02d}_best.pt')
    if not os.path.isfile(ckpt):
        raise FileNotFoundError(f'Missing best checkpoint for Fold {fold}: {ckpt}')
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    model.eval()

    mit_sse, ort_sse = 0.0, 0.0
    n_mit, n_ort = 0, 0
    rows = []
    with torch.no_grad():
        for x_b, m_b, y_b, i_b, global_idx_b in loader:
            x_b, m_b = x_b.to(DEVICE), m_b.to(DEVICE)
            y_b, i_b = y_b.to(DEVICE), i_b.to(DEVICE)
            x_hat, x_c, _, _, _ = model(x_b, m_b)
            mit_mask = (i_b > .5) & (m_b < .5)
            ort_mask = (m_b > .5) & (i_b < .5)
            mit_sse += ((x_hat - y_b).square() * mit_mask).sum().item()
            ort_sse += ((x_hat - y_b).square() * ort_mask).sum().item()
            n_mit += int(mit_mask.sum().item())
            n_ort += int(ort_mask.sum().item())
            preds = x_c.cpu().numpy()
            targets = i_b.cpu().numpy()
            y = y_b.cpu().numpy()
            for b, idx_t in enumerate(global_idx_b):
                idx = int(idx_t)
                for d in range(D_IN):
                    gaps = gap_lengths(targets[b, :, d])
                    for t in np.flatnonzero(targets[b, :, d] > .5):
                        if not np.isfinite(truth[idx, t, d]):
                            continue
                        yt, yp = float(y[b,t,d]), float(preds[b,t,d])
                        g = int(gaps[t])
                        rows.append(dict(fold=fold, split='val', segment_idx=idx,
                                         patient_id=int(seg_id[idx]), filename=str(filenames[idx]),
                                         time_idx=int(t), target=str(target_names[d]),
                                         y_true=yt, y_pred=yp, abs_err=abs(yp-yt),
                                         gap_len=g, gap_region=gap_region(g), model=MODEL_NAME))

    if n_mit == 0:
        raise ValueError(f'Fold {fold}: no eligible validation targets')
    mit_mse = mit_sse / n_mit
    ort_mse = ort_sse / n_ort if n_ort else np.nan
    total_loss = LAMBDA_MIT * mit_mse + LAMBDA_ORT * (ort_mse if n_ort else 0.0)
    all_loss.append(dict(fold=fold, model=MODEL_NAME, val_total_mse=total_loss,
                         val_mit_mse=mit_mse, val_ort_mse=ort_mse,
                         n_val_patients=len(vals), n_val_segments=len(indices),
                         n_target_points=n_mit))
    df = pd.DataFrame(rows)
    if df.empty: raise ValueError(f'Fold {fold}: no valid validation predictions')
    df.to_csv(os.path.join(OUT_DIR, f'fold{fold:02d}_validation_predictions.csv'), index=False)
    all_rows.append(df)
    for target_name, sub in df.groupby('target'):
        all_fold_metrics.append(dict(fold=fold, target=target_name, **metrics(sub)))
    for (target_name, category), sub in df.dropna(subset=['gap_region']).groupby(['target','gap_region']):
        all_gap_metrics.append(dict(fold=fold, target=target_name, gap_region=category, **metrics(sub)))
    print(f'Fold {fold}: validation MSE={total_loss:.6f} (MIT={mit_mse:.6f}, ORT={ort_mse:.6f}), points={n_mit}')
    for target_name, sub in df.groupby('target'):
        m = metrics(sub)
        print(f'  {target_name}: MAE={m["mae"]:.5f}, RMSE={m["rmse"]:.5f}, MAPE={m["mape_pct"]:.3f}%')
    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()

val_loss_df = pd.DataFrame(all_loss)
val_loss_df.to_csv(os.path.join(OUT_DIR,'cv5_validation_losses_recomputed.csv'),index=False)
loss_summary = pd.DataFrame([dict(model=MODEL_NAME,
    val_mse_mean=val_loss_df.val_total_mse.mean(),
    val_mse_std=val_loss_df.val_total_mse.std(ddof=1), n_folds=len(val_loss_df))])
loss_summary.to_csv(os.path.join(OUT_DIR,'cv5_validation_loss_summary_mean_std.csv'),index=False)
val_pred_df = pd.concat(all_rows,ignore_index=True)
val_pred_df.to_csv(os.path.join(OUT_DIR,'cv5_validation_predictions_pooled.csv'),index=False)
val_pred_df.dropna(subset=['gap_region']).to_csv(os.path.join(OUT_DIR,'cv5_validation_predictions_pooled_with_gap_region.csv'),index=False)
fold_metrics = pd.DataFrame(all_fold_metrics)
fold_metrics.to_csv(os.path.join(OUT_DIR,'cv5_validation_fold_metrics.csv'),index=False)
gap_metrics = pd.DataFrame(all_gap_metrics)
gap_metrics.to_csv(os.path.join(OUT_DIR,'cv5_validation_fold_metrics_by_gap_region.csv'),index=False)
summary = fold_metrics.groupby('target',as_index=False).agg(
    mae_mean=('mae','mean'),mae_std=('mae','std'),
    rmse_mean=('rmse','mean'),rmse_std=('rmse','std'),
    mape_mean_pct=('mape_pct','mean'),mape_std_pct=('mape_pct','std'),n_folds=('fold','count'))
summary.to_csv(os.path.join(OUT_DIR,'cv5_validation_metrics_mean_std.csv'),index=False)
pooled = pd.DataFrame([dict(target=k,**metrics(s)) for k,s in val_pred_df.groupby('target')])
pooled.to_csv(os.path.join(OUT_DIR,'cv5_validation_pooled_metrics.csv'),index=False)
pooled_gap = pd.DataFrame([dict(target=k[0],gap_region=k[1],**metrics(s))
    for k,s in val_pred_df.dropna(subset=['gap_region']).groupby(['target','gap_region'])])
pooled_gap.to_csv(os.path.join(OUT_DIR,'cv5_validation_pooled_metrics_by_gap_region.csv'),index=False)
gap_summary = gap_metrics.groupby(['target','gap_region'],as_index=False).agg(
    mae_mean=('mae','mean'),mae_std=('mae','std'),
    rmse_mean=('rmse','mean'),rmse_std=('rmse','std'),
    mape_mean_pct=('mape_pct','mean'),mape_std_pct=('mape_pct','std'),
    n_folds=('fold','count'),total_points=('n_points','sum'))
gap_summary.to_csv(os.path.join(OUT_DIR,'cv5_validation_metrics_mean_std_by_gap_region.csv'),index=False)
print('\nFive-fold validation MSE:')
print(val_loss_df.to_string(index=False))
print('\nValidation MSE mean ± SD: {:.6f} ± {:.6f}'.format(loss_summary.val_mse_mean.iloc[0],loss_summary.val_mse_std.iloc[0]))
print('\nValidation imputation metrics mean ± SD:')
print(summary.to_string(index=False))
print('\nPooled validation imputation metrics:')
print(pooled.to_string(index=False))
print('\nPooled validation imputation metrics by gap region:')
print(pooled_gap.to_string(index=False))
print('\nValidation metrics mean ± SD by gap region:')
print(gap_summary.to_string(index=False))
print('\nSaved validation files under:', OUT_DIR)

Device: cuda
Using saved patient splits: /content/drive/MyDrive/aim3/mtsit_original_aligned_1e-3_64/cv5_patient_split_summary.csv
Fold 1: validation MSE=15.221476 (MIT=15.221476, ORT=47.384541), points=9917
  channel_0: MAE=2.30714, RMSE=3.65403, MAPE=3.843%
  channel_1: MAE=3.25925, RMSE=4.66888, MAPE=5.064%
Fold 2: validation MSE=13.592446 (MIT=13.592446, ORT=10.980025), points=13337
  channel_0: MAE=1.86890, RMSE=2.99642, MAPE=2.921%
  channel_1: MAE=3.05309, RMSE=5.30160, MAPE=5.292%
Fold 3: validation MSE=44.834028 (MIT=44.834028, ORT=66.646426), points=19241
  channel_0: MAE=2.10874, RMSE=3.50916, MAPE=3.629%
  channel_1: MAE=4.17282, RMSE=12.06475, MAPE=5.928%
Fold 4: validation MSE=21.497633 (MIT=21.497633, ORT=84.197700), points=13482
  channel_0: MAE=2.52823, RMSE=4.75370, MAPE=5.241%
  channel_1: MAE=2.79121, RMSE=4.29927, MAPE=4.948%
Fold 5: validation MSE=43.612527 (MIT=43.612527, ORT=93.910637), points=8249
  channel_0: MAE=1.88373, RMSE=2.76021, MAPE=3.459%
  channel_1: 

### Model testing

In [7]:
import os
import numpy as np
import pandas as pd

# ============================================================
# Config — MTSIT
# ============================================================
PRED_CSV = os.path.join(
    OUT_DIR, "cv5_test_predictions_pooled.csv"
)

EPS = 1e-8

regions_order = [
    "<1 min",
    "[1,10) min",
    "[10,20) min",
    "[20,30] min",
    "ALL"
]

MODEL_NAME = "MTSIT_OriginalAligned"

# ============================================================
# Load test predictions
# ============================================================

pred_df = pd.read_csv(PRED_CSV)

if "model" not in pred_df.columns:
    pred_df["model"] = MODEL_NAME

for col in ["gap_len", "y_true", "y_pred"]:
    pred_df[col] = pd.to_numeric(
        pred_df[col], errors="coerce"
    )

pred_df = pred_df[
    pred_df["gap_len"].between(1, 360)
    & np.isfinite(pred_df["y_true"])
    & np.isfinite(pred_df["y_pred"])
].copy()

print("Total test prediction points:", len(pred_df))
print("Number of folds:", pred_df["fold"].nunique())

pred_df["gap_region"] = pd.cut(
    pred_df["gap_len"],
    bins=[0, 11, 119, 239, 360],
    labels=regions_order[:-1],
    include_lowest=True
)


def calculate_metrics(df):

    yt = df["y_true"].to_numpy(dtype=float)
    yp = df["y_pred"].to_numpy(dtype=float)

    abs_err = np.abs(yp - yt)
    valid_mape = np.abs(yt) > EPS

    return pd.Series({
        "MAE": abs_err.mean(),

        "MAPE_pct": (
            100 * np.mean(
                abs_err[valid_mape] /
                np.abs(yt[valid_mape])
            )
            if valid_mape.any() else np.nan
        ),

        "n_points": len(yt),
        "n_mape_points": int(valid_mape.sum())
    })


fold_gap = (
    pred_df
    .groupby(
        ["fold", "target", "model", "gap_region"],
        observed=True
    )[["y_true", "y_pred"]]
    .apply(calculate_metrics)
    .reset_index()
)

fold_all = (
    pred_df
    .groupby(
        ["fold", "target", "model"],
        observed=True
    )[["y_true", "y_pred"]]
    .apply(calculate_metrics)
    .reset_index()
)

fold_all["gap_region"] = "ALL"

foldwise = pd.concat(
    [fold_gap, fold_all],
    ignore_index=True
)

foldwise["gap_region"] = pd.Categorical(
    foldwise["gap_region"],
    categories=regions_order,
    ordered=True
)

foldwise = foldwise.sort_values(
    ["target", "model", "gap_region", "fold"]
).reset_index(drop=True)


# ============================================================
# Mean ± STD across five folds
# ============================================================

summary = (
    foldwise
    .groupby(
        ["target", "model", "gap_region"],
        observed=True
    )
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        MAPE_mean_pct=("MAPE_pct", "mean"),
        MAPE_std_pct=("MAPE_pct", "std"),

        n_folds_MAE=("MAE", "count"),
        n_folds_MAPE=("MAPE_pct", "count"),

        total_points=("n_points", "sum")
    )
    .reset_index()
)


# ============================================================
# Manuscript-ready table
# Same format as CSDI / LSCD
# ============================================================

formatted = summary.copy()

for name, mean_col, std_col in [
    ("MAE", "MAE_mean", "MAE_std"),
    ("MAPE(%)", "MAPE_mean_pct", "MAPE_std_pct")
]:

    formatted[name] = formatted.apply(
        lambda r: (
            f"{r[mean_col]:.2f} ± {r[std_col]:.2f}"
            if pd.notna(r[std_col])
            else (
                f"{r[mean_col]:.2f} ± NA"
                if pd.notna(r[mean_col])
                else "NA"
            )
        ),
        axis=1
    )

paper = formatted.pivot(
    index=["target", "model"],
    columns="gap_region",
    values=["MAE", "MAPE(%)"]
)

paper = paper.swaplevel(0, 1, axis=1)

paper = paper.reindex(
    columns=pd.MultiIndex.from_product(
        [regions_order, ["MAE", "MAPE(%)"]]
    )
)

paper.columns = [
    f"{gap} | {metric}"
    for gap, metric in paper.columns
]

paper = paper.reset_index()


# ============================================================
# Pooled metrics
# ============================================================

pool_gap = (
    pred_df
    .groupby(
        ["target", "model", "gap_region"],
        observed=True
    )[["y_true", "y_pred"]]
    .apply(calculate_metrics)
    .reset_index()
)

pool_all = (
    pred_df
    .groupby(
        ["target", "model"],
        observed=True
    )[["y_true", "y_pred"]]
    .apply(calculate_metrics)
    .reset_index()
)

pool_all["gap_region"] = "ALL"

pooled = pd.concat(
    [pool_gap, pool_all],
    ignore_index=True
)


# ============================================================
# Save results
# ============================================================

outputs = {
    "mtsit_paper4_foldwise.csv": foldwise,
    "mtsit_paper4_mean_std.csv": summary,
    "mtsit_paper4_pooled.csv": pooled,
    "mtsit_paper4_paper_table.csv": paper,
}

for filename, df in outputs.items():
    path = os.path.join(OUT_DIR, filename)
    df.to_csv(path, index=False)
    print("Saved:", path)


# ============================================================
# Print results
# ============================================================

print("\nMTSIT FIVE-FOLD MEAN ± STD:")
print(paper.to_string(index=False))

print("\nMTSIT FOLDWISE METRICS:")
print(foldwise.to_string(index=False))

print("\nMTSIT POOLED METRICS:")
print(pooled.to_string(index=False))

Total test prediction points: 144428
Number of folds: 5
Saved: /content/drive/MyDrive/aim3/mtsit_original_aligned_1e-3_64/mtsit_paper4_foldwise.csv
Saved: /content/drive/MyDrive/aim3/mtsit_original_aligned_1e-3_64/mtsit_paper4_mean_std.csv
Saved: /content/drive/MyDrive/aim3/mtsit_original_aligned_1e-3_64/mtsit_paper4_pooled.csv
Saved: /content/drive/MyDrive/aim3/mtsit_original_aligned_1e-3_64/mtsit_paper4_paper_table.csv

MTSIT FIVE-FOLD MEAN ± STD:
   target                 model <1 min | MAE <1 min | MAPE(%) [1,10) min | MAE [1,10) min | MAPE(%) [10,20) min | MAE [10,20) min | MAPE(%) [20,30] min | MAE [20,30] min | MAPE(%)   ALL | MAE ALL | MAPE(%)
channel_0 MTSIT_OriginalAligned  2.02 ± 0.17      3.47 ± 0.29      2.19 ± 0.28          4.13 ± 0.94       2.18 ± 0.74           3.91 ± 1.58       2.19 ± 0.54           3.92 ± 1.67 2.14 ± 0.35   3.85 ± 0.88
channel_1 MTSIT_OriginalAligned  3.45 ± 0.53      6.15 ± 1.75      3.76 ± 0.68          5.95 ± 0.88       2.85 ± 0.69           4.54 ±